# CPSTN (Testing) - Draft 4

**Purpose:** standalone, efficient Colab/T4 testing notebook for the **CPSTN-Draft4** training pipeline.

This notebook replaces the faulty `CPSTN_Enhanced_PEMS_BAY_Test` pipeline.

### What is fixed

- Reconstructs the **exact CPSTN-Draft4 architecture** used by `CPSTN(Training)-Draft4`.
- Loads the scaler and normalized graph directly from the checkpoint.
- Uses the same `0 -> NaN -> interpolation -> ffill/bfill` preprocessing.
- Uses the same **70% train / 10% validation / 20% test** chronological split.
- Uses the same `12 -> 12` history/forecast convention stored in the checkpoint.
- Preserves sensor-ID ordering from the checkpoint.
- Uses the checkpoint's graph instead of downloading/rebuilding a potentially mismatched graph.
- Uses `torch.inference_mode()` and CUDA batching for fast T4 inference.
- Automatically supports **METR-LA or PEMS-BAY**, based on the checkpoint.
- Produces overall, horizon-wise, per-sensor, and persistence-baseline metrics.
- Saves predictions, metrics, metadata, and plots.
- Includes integrity checks designed to catch the architecture/graph/scaler mismatches that caused the previous testing notebook to fail.

> **Important:** testing cannot improve a trained model. It only measures the checkpoint honestly. The provided Draft4 checkpoint is METR-LA, so this notebook will automatically test METR-LA when that checkpoint is used. A PEMS-BAY checkpoint can be tested with the same notebook if you upload one.


In [ ]:
# 1. Colab/T4 environment

!nvidia-smi

import os
import gc
import json
import time
import pickle
import hashlib
import shutil
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required. In Colab use: Runtime -> Change runtime type -> GPU."
    )

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)


## 2. Configuration and checkpoint selection

The notebook first looks for the supplied Draft4 checkpoint. If it is not present in `/content`, a Colab upload dialog is opened. You can upload either a `.pt` or `.pth` checkpoint created by the Draft4 training notebook.

In [ ]:
# 2. Locate/upload the trained checkpoint

ROOT = Path("/content/cpstn_testing_draft4")
ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Common names, including the checkpoint supplied with the Draft4 training run.
candidate_paths = [
    Path("/content/CPSTN_Draft4_METR-LA_best.pt"),
    Path("/content/CPSTN_Draft4_PEMS-BAY_best.pt"),
    Path("/content/CPSTN_Enhanced_PEMS_BAY_best.pt"),
    Path("/content/cpstn_draft4/CPSTN_Draft4_METR-LA_best.pt"),
    Path("/content/cpstn_draft4/CPSTN_Draft4_PEMS-BAY_best.pt"),
]

MODEL_PATH = next((p for p in candidate_paths if p.exists()), None)

if MODEL_PATH is None:
    from google.colab import files
    print("Upload the Draft4 trained checkpoint (.pt / .pth).")
    uploaded = files.upload()
    pt_files = [Path(name) for name in uploaded if name.lower().endswith((".pt", ".pth"))]
    if not pt_files:
        raise RuntimeError("No .pt/.pth checkpoint was uploaded.")
    MODEL_PATH = pt_files[0]

print("Checkpoint:", MODEL_PATH)
print("Size:", round(MODEL_PATH.stat().st_size / 1024**2, 2), "MB")


In [ ]:
# 3. Load checkpoint FIRST
# This prevents the test notebook from inventing preprocessing/model settings.

try:
    ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(MODEL_PATH, map_location="cpu")

required = [
    "model_state_dict", "dataset", "num_nodes", "input_steps", "horizon",
    "input_channels", "hidden", "dropout", "mean", "std",
    "adjacency_normalized", "sensor_ids"
]
missing = [k for k in required if k not in ckpt]
if missing:
    raise RuntimeError(
        "This is not a compatible CPSTN-Draft4 checkpoint. Missing: " + str(missing)
    )

DATASET = str(ckpt["dataset"])
N = int(ckpt["num_nodes"])
INPUT_STEPS = int(ckpt["input_steps"])
HORIZON = int(ckpt["horizon"])
INPUT_CHANNELS = int(ckpt["input_channels"])
HIDDEN = int(ckpt["hidden"])
DROPOUT = float(ckpt["dropout"])

mean = np.asarray(ckpt["mean"], dtype=np.float32)
std = np.asarray(ckpt["std"], dtype=np.float32)
A_norm = np.asarray(ckpt["adjacency_normalized"], dtype=np.float32)
sensor_ids = [str(x) for x in ckpt["sensor_ids"]]

EXPECTED_NODES = {"METR-LA": 207, "PEMS-BAY": 325}
if DATASET not in EXPECTED_NODES:
    raise RuntimeError(f"Unsupported Draft4 dataset in checkpoint: {DATASET}")

if N != EXPECTED_NODES[DATASET]:
    raise RuntimeError(f"{DATASET}: checkpoint says {N} nodes, expected {EXPECTED_NODES[DATASET]}.")

if INPUT_CHANNELS != 3:
    raise RuntimeError(f"Draft4 expects 3 input channels, checkpoint has {INPUT_CHANNELS}.")

if mean.shape != (N,) or std.shape != (N,):
    raise RuntimeError("Checkpoint scaler shape does not match node count.")

if A_norm.shape != (N, N):
    raise RuntimeError("Checkpoint normalized adjacency shape is invalid.")

if not np.isfinite(mean).all() or not np.isfinite(std).all() or not np.isfinite(A_norm).all():
    raise RuntimeError("Checkpoint contains non-finite scaler/graph values.")

if len(sensor_ids) != N or len(set(sensor_ids)) != N:
    raise RuntimeError("Checkpoint sensor IDs are missing or duplicated.")

print("=" * 70)
print("CHECKPOINT CONFIGURATION")
print("=" * 70)
print("Dataset          :", DATASET)
print("Sensors          :", N)
print("Input steps      :", INPUT_STEPS)
print("Forecast horizon :", HORIZON)
print("Input channels   :", INPUT_CHANNELS)
print("Hidden channels  :", HIDDEN)
print("Dropout          :", DROPOUT)
print("Best epoch       :", ckpt.get("epoch"))
print("Best val loss    :", ckpt.get("best_val_loss"))
print("Model name       :", ckpt.get("model_name"))
print("Graph backend    :", ckpt.get("graph_backend"))
print("=" * 70)


In [ ]:
# 4. Dataset URLs
# Only the traffic data is downloaded here. The checkpoint already contains the
# exact normalized graph, so the test pipeline does not rebuild graph normalization.

DATA_CFG = {
    "METR-LA": {
        "file": DATA_DIR / "metr-la.h5",
        "url": "https://drive.google.com/uc?export=download&id=1pAGRfzMx6K9WWsfDcD1NMbIif0T0saFC",
        "drive_id": "1pAGRfzMx6K9WWsfDcD1NMbIif0T0saFC",
        "expected_nodes": 207,
    },
    "PEMS-BAY": {
        "file": DATA_DIR / "pems-bay.h5",
        "url": "https://drive.google.com/uc?export=download&id=1wD-mHlqAb2mtHOe_68fZvDh1LpDegMMq",
        "drive_id": "1wD-mHlqAb2mtHOe_68fZvDh1LpDegMMq",
        "expected_nodes": 325,
    },
}
cfg = DATA_CFG[DATASET]

try:
    import gdown
except ImportError:
    !pip -q install gdown
    import gdown

def download_dataset():
    path = Path(cfg["file"])
    if path.exists() and path.stat().st_size > 0:
        print("Using cached dataset:", path)
        return path

    print("Downloading", DATASET, "traffic data...")
    gdown.download(
        id=cfg["drive_id"],
        output=str(path),
        quiet=False,
    )
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError("Dataset download failed.")
    return path

H5_PATH = download_dataset()
print("Dataset file:", H5_PATH)
print("Size:", round(H5_PATH.stat().st_size / 1024**2, 2), "MB")


In [ ]:
# 5. Load traffic data using the SAME repair logic as Draft4 training

try:
    df = pd.read_hdf(H5_PATH, key="speed")
except (KeyError, ValueError):
    df = pd.read_hdf(H5_PATH)

df.index = pd.to_datetime(df.index)
df = df.sort_index()
df.columns = [str(c) for c in df.columns]

if df.shape[1] != N:
    raise RuntimeError(
        f"Dataset/checkpoint mismatch: data has {df.shape[1]} sensors, checkpoint has {N}."
    )

# Exact Draft4 missing-value treatment.
df = df.apply(pd.to_numeric, errors="coerce")
missing_before = int(df.isna().sum().sum())
df = df.replace([np.inf, -np.inf], np.nan)
df = df.replace(0.0, np.nan)
df = df.interpolate(method="linear", limit_direction="both").ffill().bfill()

if df.isna().any().any():
    raise RuntimeError("NaNs remain after Draft4 preprocessing.")

speed = np.ascontiguousarray(df.to_numpy(dtype=np.float32))
timestamps = df.index

T = len(speed)
train_end = int(T * 0.70)
val_end = int(T * 0.80)

# IMPORTANT: use ONLY the train scaler saved inside the checkpoint.
speed_z = ((speed - mean[None, :]) / np.maximum(std[None, :], 1e-3)).astype(np.float32)

print("Data shape          :", speed.shape)
print("Time range          :", timestamps.min(), "->", timestamps.max())
print("Median interval     :", timestamps.to_series().diff().dropna().median())
print("Missing before fix  :", missing_before)
print("Train end           :", train_end)
print("Validation end      :", val_end)
print("Test rows           :", T - val_end)
print("Test interval       :", timestamps[val_end], "->", timestamps[-1])


## 6. Exact Draft4 test-window construction

The old notebook allowed the first test target to use a history window that crossed the split boundary. Draft4 training did not do that. This version follows the training notebook exactly: the first test target is at `val_end + INPUT_STEPS`, and every target remains fully inside the test segment.

In [ ]:
# 6. Build ONLY test windows.
# This keeps RAM usage much lower than rebuilding train + validation + test windows.

minutes = timestamps.hour.to_numpy() * 60 + timestamps.minute.to_numpy()
tod = (minutes / 1440.0).astype(np.float32)
tf_all = np.stack(
    [np.sin(2 * np.pi * tod), np.cos(2 * np.pi * tod)],
    axis=1
).astype(np.float32)

first_t = val_end + INPUT_STEPS
last_t = T - HORIZON

if last_t < first_t:
    raise RuntimeError("Test segment is too small for the checkpoint horizon.")

starts = np.arange(first_t, last_t + 1, dtype=np.int64)

# Vectorized window materialization.
X_speed = np.stack(
    [speed_z[t-INPUT_STEPS:t] for t in starts],
    axis=0
)
X_time = np.stack(
    [tf_all[t-INPUT_STEPS:t] for t in starts],
    axis=0
)
X_time = np.broadcast_to(
    X_time[:, :, None, :],
    (len(starts), INPUT_STEPS, N, 2)
).copy()

X = np.concatenate([X_speed[..., None], X_time], axis=-1).astype(np.float32)
Y = np.stack([speed_z[t:t+HORIZON] for t in starts], axis=0).astype(np.float32)

X = np.ascontiguousarray(X)
Y = np.ascontiguousarray(Y)

print("Test windows:", len(X))
print("X:", X.shape, X.dtype, "RAM:", round(X.nbytes / 1024**2, 1), "MB")
print("Y:", Y.shape, Y.dtype, "RAM:", round(Y.nbytes / 1024**2, 1), "MB")

assert X.shape == (len(starts), INPUT_STEPS, N, 3)
assert Y.shape == (len(starts), HORIZON, N)


In [ ]:
# 7. Exact CPSTN-Draft4 architecture
# This is copied structurally from the supplied Training-Draft4 notebook.
# Do NOT replace this architecture with the EnhancedSTGCN from the faulty notebook.

class DenseGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x, A_dense):
        # x: [B, C, T, N]
        B, C, Tm, Nn = x.shape

        z = (
            x.permute(0, 2, 1, 3)
             .contiguous()
             .reshape(B * Tm * C, Nn)
        )

        # Exact Draft4 graph backend: FP32 dense graph propagation.
        with torch.autocast(device_type="cuda", enabled=False):
            z = torch.matmul(A_dense.float(), z.float().T).T

        z = z.to(dtype=x.dtype)
        z = (
            z.reshape(B, Tm, C, Nn)
             .permute(0, 2, 1, 3)
             .contiguous()
        )
        return self.proj(z)


class TemporalGate(nn.Module):
    def __init__(self, in_channels, out_channels, dilation=1, dropout=0.1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            2 * out_channels,
            kernel_size=(3, 1),
            padding=(dilation, 0),
            dilation=(dilation, 1),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a, b = self.conv(x).chunk(2, dim=1)
        return self.dropout(torch.tanh(a) * torch.sigmoid(b))


class CPSTNBlock(nn.Module):
    def __init__(self, channels, dropout=0.1, dilation=1):
        super().__init__()
        self.temporal = TemporalGate(
            channels, channels, dilation=dilation, dropout=dropout
        )
        self.graph = DenseGraphConv(channels, channels)
        self.norm = nn.BatchNorm2d(channels)
        self.ff = nn.Conv2d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, A_dense):
        residual = x
        x = self.temporal(x)
        x = F.gelu(self.graph(x, A_dense))
        x = self.ff(x)
        x = self.dropout(self.norm(x))
        return F.gelu(x + residual)


class CPSTN(nn.Module):
    def __init__(self, input_channels=3, hidden=32, horizon=12, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Conv2d(input_channels, hidden, kernel_size=1)
        self.block1 = CPSTNBlock(hidden, dropout, dilation=1)
        self.block2 = CPSTNBlock(hidden, dropout, dilation=2)
        self.block3 = CPSTNBlock(hidden, dropout, dilation=4)
        self.head = nn.Sequential(
            nn.Conv2d(hidden, hidden, kernel_size=1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv2d(hidden, horizon, kernel_size=1),
        )

    def forward(self, x, A_dense):
        x = x.permute(0, 3, 1, 2).contiguous()
        x = self.input_proj(x)
        x = self.block1(x, A_dense)
        x = self.block2(x, A_dense)
        x = self.block3(x, A_dense)
        x = x[:, :, -1:, :]
        return self.head(x).squeeze(2)


A_tensor = torch.from_numpy(A_norm).to(device=device, dtype=torch.float32)

model = CPSTN(
    input_channels=INPUT_CHANNELS,
    hidden=HIDDEN,
    horizon=HORIZON,
    dropout=DROPOUT,
).to(device)

model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()

params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model parameters:", f"{params:,}")
print("Graph:", tuple(A_tensor.shape), A_tensor.dtype, A_tensor.device)
print("State dict load: PASS")


In [ ]:
# 8. Mandatory model smoke test before the full run

smoke_batch = min(2, len(X))
with torch.inference_mode():
    smoke_x = torch.from_numpy(X[:smoke_batch]).to(device, non_blocking=True)
    smoke_pred = model(smoke_x, A_tensor)

expected_shape = (smoke_batch, HORIZON, N)
assert tuple(smoke_pred.shape) == expected_shape
assert torch.isfinite(smoke_pred).all()

del smoke_x, smoke_pred
torch.cuda.empty_cache()

print("Smoke-test output:", expected_shape)
print("Finite output     : PASS")
print("Architecture test : PASS")


## 7. Fast T4 inference

The default batch size is intentionally conservative. The loop automatically retries with a smaller batch if CUDA runs out of memory, rather than turning the notebook into a flaming pile of tensors.

In [ ]:
# 9. Fast full-test inference with automatic CUDA OOM fallback

def run_inference(batch_size):
    pred_chunks = []
    start_time = time.time()

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(X), batch_size),
            desc=f"Testing (batch={batch_size})"
        ):
            xb = torch.from_numpy(X[start:start+batch_size]).to(
                device, non_blocking=True
            )
            pb = model(xb, A_tensor)
            pred_chunks.append(pb.float().cpu().numpy())
            del xb, pb

    torch.cuda.synchronize()
    elapsed = time.time() - start_time
    return np.concatenate(pred_chunks, axis=0), elapsed


preferred_batch = 256 if N <= 325 else 128
batch_candidates = [preferred_batch, 128, 64, 32, 16]

pred_z = None
elapsed = None

for bs in batch_candidates:
    if pred_z is not None:
        break
    try:
        gc.collect()
        torch.cuda.empty_cache()
        pred_z, elapsed = run_inference(bs)
        USED_BATCH_SIZE = bs
    except torch.cuda.OutOfMemoryError:
        print(f"CUDA OOM at batch size {bs}; retrying smaller...")
        pred_z = None
        gc.collect()
        torch.cuda.empty_cache()

if pred_z is None:
    raise RuntimeError("Inference still ran out of GPU memory at the smallest batch size.")

true_z = Y

assert pred_z.shape == true_z.shape == (len(X), HORIZON, N)
assert np.isfinite(pred_z).all()

print("=" * 70)
print("INFERENCE COMPLETE")
print("=" * 70)
print("Batch size       :", USED_BATCH_SIZE)
print("Prediction shape :", pred_z.shape)
print("Inference time   :", round(elapsed, 2), "seconds")
print("Windows/sec      :", round(len(X) / max(elapsed, 1e-9), 2))
print("GPU peak memory  :", round(torch.cuda.max_memory_allocated() / 1024**3, 3), "GB")
print("=" * 70)


In [ ]:
# 10. Convert predictions back to original speed units

pred = pred_z * std[None, None, :] + mean[None, None, :]
true = true_z * std[None, None, :] + mean[None, None:]

assert pred.shape == true.shape
assert np.isfinite(pred).all()
assert np.isfinite(true).all()

print("Prediction/target:", pred.shape)
print("Pred range:", float(pred.min()), "->", float(pred.max()))
print("True range:", float(true.min()), "->", float(true.max()))


In [ ]:
# 11. Overall metrics
# MAPE uses the same practical denominator floor used by Draft4 training evaluation.

def compute_metrics(p, y):
    err = p - y
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    mape = np.mean(np.abs(err) / np.maximum(np.abs(y), 5.0)) * 100.0
    return float(mae), float(rmse), float(mape)

mae, rmse, mape = compute_metrics(pred, true)

print("=" * 70)
print(f"{DATASET} | CPSTN-Draft4 TEST")
print("=" * 70)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")
print("=" * 70)


In [ ]:
# 12. Horizon-wise metrics: 5, 10, ..., 60 minutes for the default Draft4 horizon

rows = []
for h in range(HORIZON):
    h_mae, h_rmse, h_mape = compute_metrics(
        pred[:, h, :],
        true[:, h, :]
    )
    rows.append({
        "horizon_step": h + 1,
        "horizon_minutes": (h + 1) * 5,
        "MAE": h_mae,
        "RMSE": h_rmse,
        "MAPE_percent": h_mape,
    })

horizon_df = pd.DataFrame(rows)
display(horizon_df)

plt.figure(figsize=(10, 4))
plt.plot(horizon_df["horizon_minutes"], horizon_df["MAE"], marker="o", label="MAE")
plt.plot(horizon_df["horizon_minutes"], horizon_df["RMSE"], marker="s", label="RMSE")
plt.xlabel("Forecast horizon (minutes)")
plt.ylabel("Error")
plt.title(f"{DATASET}: CPSTN-Draft4 horizon-wise test error")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 13. Persistence baseline
# Last observed speed repeated over the complete forecast horizon.

last_input_z = X[:, -1, :, 0]
baseline_z = np.repeat(last_input_z[:, None, :], HORIZON, axis=1)

baseline = baseline_z * std[None, None, :] + mean[None, None, :]

b_mae, b_rmse, b_mape = compute_metrics(baseline, true)

comparison_df = pd.DataFrame({
    "Model": ["CPSTN-Draft4", "Persistence"],
    "MAE": [mae, b_mae],
    "RMSE": [rmse, b_rmse],
    "MAPE_percent": [mape, b_mape],
})

display(comparison_df)

print("Persistence baseline:")
print(f"MAE  : {b_mae:.4f}")
print(f"RMSE : {b_rmse:.4f}")
print(f"MAPE : {b_mape:.2f}%")


In [ ]:
# 14. Per-sensor metrics
sensor_mae = np.mean(np.abs(pred - true), axis=(0, 1))
sensor_rmse = np.sqrt(np.mean((pred - true) ** 2, axis=(0, 1)))

sensor_df = pd.DataFrame({
    "sensor_id": sensor_ids,
    "MAE": sensor_mae,
    "RMSE": sensor_rmse,
}).sort_values("MAE", ascending=False)

print("Worst 10 sensors by MAE:")
display(sensor_df.head(10))

print("Best 10 sensors by MAE:")
display(sensor_df.tail(10).sort_values("MAE"))


In [ ]:
# 15. Forecast visualization for a representative sensor

sample_idx = min(100, len(pred) - 1)
sensor_idx = 0

# Recover the history corresponding to this exact test sample.
history_z = X[sample_idx, :, sensor_idx, 0]
history = history_z * std[sensor_idx] + mean[sensor_idx]

future_true = true[sample_idx, :, sensor_idx]
future_pred = pred[sample_idx, :, sensor_idx]

history_x = np.arange(INPUT_STEPS)
future_x = np.arange(INPUT_STEPS, INPUT_STEPS + HORIZON)

plt.figure(figsize=(11, 4))
plt.plot(history_x, history, marker="o", label="History")
plt.plot(future_x, future_true, marker="o", label="True future")
plt.plot(future_x, future_pred, marker="x", label="CPSTN-Draft4")
plt.axvline(INPUT_STEPS - 0.5, linestyle="--", alpha=0.5)
plt.xlabel("5-minute step")
plt.ylabel("Speed")
plt.title(f"{DATASET}: sensor {sensor_ids[sensor_idx]} forecast")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 16. Spatial sanity checks
# Compare one test sample across all sensors at 5, 30 and final forecast horizon.

for h_idx in sorted(set([0, min(5, HORIZON - 1), HORIZON - 1])):
    plt.figure(figsize=(11, 3.8))
    plt.plot(true[sample_idx, h_idx], label="True")
    plt.plot(pred[sample_idx, h_idx], label="Predicted", alpha=0.85)
    plt.xlabel("Sensor index (checkpoint-aligned)")
    plt.ylabel("Speed")
    plt.title(
        f"{DATASET}: sample {sample_idx}, +{(h_idx + 1) * 5} min"
    )
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# 17. Export all useful testing artifacts

pred_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft4_test_predictions.npz"
metrics_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft4_test_metrics.csv"
horizon_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft4_horizon_metrics.csv"
sensor_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft4_sensor_metrics.csv"
metadata_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft4_test_metadata.json"

np.savez_compressed(
    pred_path,
    prediction=pred.astype(np.float32),
    target=true.astype(np.float32),
    prediction_z=pred_z.astype(np.float32),
    target_z=true_z.astype(np.float32),
    sample_start_indices=starts.astype(np.int64),
)

comparison_df.to_csv(metrics_path, index=False)
horizon_df.to_csv(horizon_path, index=False)
sensor_df.to_csv(sensor_path, index=False)

metadata = {
    "dataset": DATASET,
    "model": str(ckpt.get("model_name", "CPSTN-Draft4")),
    "checkpoint": str(MODEL_PATH),
    "checkpoint_epoch": ckpt.get("epoch"),
    "best_validation_loss": ckpt.get("best_val_loss"),
    "num_nodes": N,
    "input_steps": INPUT_STEPS,
    "horizon": HORIZON,
    "input_channels": INPUT_CHANNELS,
    "hidden": HIDDEN,
    "dropout": DROPOUT,
    "test_split": "20% chronological",
    "train_fraction": 0.70,
    "validation_fraction": 0.10,
    "test_start_row": int(val_end),
    "test_first_target_row": int(first_t),
    "test_last_target_start_row": int(last_t),
    "num_test_windows": int(len(X)),
    "batch_size": int(USED_BATCH_SIZE),
    "inference_seconds": float(elapsed),
    "test_mae": mae,
    "test_rmse": rmse,
    "test_mape_percent": mape,
    "persistence_mae": b_mae,
    "persistence_rmse": b_rmse,
    "persistence_mape_percent": b_mape,
    "sensor_ids": sensor_ids,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
for p in sorted(EXPORT_DIR.iterdir()):
    print(" -", p.name, f"({p.stat().st_size / 1024**2:.2f} MB)")


## 9. Final integrity checks

These checks are deliberately strict. A pretty MAE is useless if the graph, sensor ordering, scaler, architecture, or test-window indexing is wrong. Humanity has somehow made this a recurring machine-learning tradition.

In [ ]:
# 18. Final integrity report

assert MODEL_PATH.exists()
assert DATASET in {"METR-LA", "PEMS-BAY"}
assert N == EXPECTED_NODES[DATASET]
assert len(sensor_ids) == N
assert A_norm.shape == (N, N)
assert np.isfinite(A_norm).all()
assert np.allclose(A_norm, A_norm.T, atol=1e-6)

assert X.shape[1:] == (INPUT_STEPS, N, 3)
assert Y.shape[1:] == (HORIZON, N)
assert pred.shape == true.shape == (len(X), HORIZON, N)

assert np.isfinite(pred).all()
assert np.isfinite(true).all()
assert np.isfinite([mae, rmse, mape, b_mae, b_rmse, b_mape]).all()

assert len(starts) == len(X)
assert starts[0] == val_end + INPUT_STEPS
assert starts[-1] == T - HORIZON

print("=" * 78)
print("CPSTN DRAFT 4 TESTING INTEGRITY CHECK: PASS")
print("=" * 78)
print("Dataset              :", DATASET)
print("Sensors              :", N)
print("Input -> horizon     :", f"{INPUT_STEPS} -> {HORIZON}")
print("Checkpoint epoch     :", ckpt.get("epoch"))
print("Best validation loss :", ckpt.get("best_val_loss"))
print("Test windows         :", len(X))
print("Test MAE             :", f"{mae:.4f}")
print("Test RMSE            :", f"{rmse:.4f}")
print("Test MAPE            :", f"{mape:.2f}%")
print("Persistence MAE      :", f"{b_mae:.4f}")
print("Persistence RMSE     :", f"{b_rmse:.4f}")
print("Persistence MAPE     :", f"{b_mape:.2f}%")
print("Graph shape          :", A_norm.shape)
print("Graph symmetry       : PASS")
print("Sensor-ID integrity  : PASS")
print("Test-window indexing : PASS")
print("Prediction shape     :", pred.shape)
print("Inference batch      :", USED_BATCH_SIZE)
print("Inference time       :", f"{elapsed:.2f}s")
print("Exports              :", EXPORT_DIR)
print("=" * 78)


In [ ]:
# 19. Optional: create one ZIP containing all testing results

zip_base = ROOT / f"CPSTN(Testing)-Draft4_{DATASET}_outputs"

if EXPORT_DIR.exists():
    shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=EXPORT_DIR
    )

print("ZIP created:", str(zip_base) + ".zip")

# Uncomment in Colab when you want the browser download:
# from google.colab import files
# files.download(str(zip_base) + ".zip")


## Expected workflow in Colab

1. **Runtime → Change runtime type → T4 GPU**
2. Run cells from top to bottom.
3. Upload `CPSTN_Draft4_METR-LA_best.pt` when requested, or place the checkpoint in `/content`.
4. The notebook automatically detects `METR-LA` from the checkpoint.
5. It downloads only the matching traffic data.
6. It reconstructs the exact Draft4 model.
7. It runs the complete chronological test set.
8. Check **MAE / RMSE / MAPE**, horizon metrics, persistence comparison, and the final integrity report.
9. The generated ZIP contains the test predictions and CSV/JSON reports.

### Important PEMS-BAY note

The supplied checkpoint in this task is **METR-LA (207 sensors)**. It cannot legitimately be used to test PEMS-BAY (325 sensors). The notebook therefore refuses a dataset/checkpoint mismatch instead of silently producing garbage metrics. For PEMS-BAY, use a checkpoint trained on PEMS-BAY with this same Draft4 architecture and preprocessing.
